# Partial-Discharge Acoustic Dataset — Analysis, Cleaning & EDA

The files in `../datasets/*.csv` are **single-channel acoustic emission recordings** of
partial-discharge (PD) activity on ceramic insulator discs under HV stress.

| Property | Value |
|---|---|
| Format | headerless CSV, 2 columns: `time_s`, `amplitude_counts` |
| Samples / file | 15,360,000 |
| Sample rate | 256 kHz (Δt = 3.90625 µs) |
| Duration | 60.0 s |
| Amplitude | signed integer ADC counts (no physical calibration provided) |

**Defect classes** (from filenames): `corona`, `dba` (dry-band arcing), `triplen_point`,
`surface_discharge`.

**Split** — the loose CSVs are the union of two OneDrive archives:
* *Two Disc Acoustic Data (**Training**)* → `corona ceramic_*`, `DBA Brown Ceramic (2 disc)_*`, `triplen point (2 ceramic disc)_*`
* *Three Disc Acoustic Data (**Testing**)* → `Corona 50KV`, `DBA 60mS 30KV`, `DBA 7.5mS 30KV`, `Middle damaged disc (surface discharge)`

This notebook: inventory → fast loading w/ cache → data-quality checks → cleaning →
EDA plots (waveforms, spectra, spectrograms, envelopes) → PD pulse detection →
per-file feature table → cleaned/windowed exports.

In [ ]:
from __future__ import annotations
import re, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal, stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (12, 4),
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 9,
})

# ---------------------------------------------------------------- configuration
DATA_DIR  = Path("../datasets")
CACHE_DIR = DATA_DIR / "_cache"        # parquet copies for fast re-loads
OUT_DIR   = DATA_DIR / "_processed"    # cleaned + windowed exports
CACHE_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

FS        = 256_000                    # Hz  (nominal, verified below)
DT        = 1.0 / FS
MAX_SECONDS = None                    # set e.g. 20 to speed up a first pass; None = full 60 s

print("datasets dir:", DATA_DIR.resolve())
print("csv files   :", len(sorted(DATA_DIR.glob("*.csv"))))

## 1 · Inventory — parse metadata from filenames

In [ ]:
def parse_meta(path: Path) -> dict:
    stem = path.stem
    low  = stem.lower()

    if   "corona"  in low:                                  cls = "corona"
    elif "dba"     in low:                                  cls = "dba"
    elif "triplen" in low:                                  cls = "triplen_point"
    elif "surface discharge" in low or "damaged disc" in low: cls = "surface_discharge"
    else:                                                   cls = "unknown"

    # two-disc archive == training material, three-disc == testing material
    is_train = any(k in low for k in ("measurement", "2 disc", "2 ceramic", "ceramic_"))
    split = "train" if is_train else "test"

    m  = re.search(r"measurement\s*(\d+)", low)
    kv = re.search(r"(\d+(?:\.\d+)?)\s*kv", low)
    ms = re.search(r"(\d+(?:\.\d+)?)\s*ms", low)

    return dict(
        name        = stem,
        cls         = cls,
        split       = split,
        measurement = int(m.group(1)) if m else pd.NA,
        kV          = float(kv.group(1)) if kv else pd.NA,
        dba_param   = f"{ms.group(1)}mS" if ms else pd.NA,
        file        = str(path),
        size_mb     = round(path.stat().st_size / 1e6, 1),
    )

inv = pd.DataFrame(parse_meta(p) for p in sorted(DATA_DIR.glob("*.csv")))
inv.index.name = "id"
inv

In [ ]:
print("files per class / split")
display(inv.pivot_table(index="cls", columns="split", values="name",
                        aggfunc="count", fill_value=0))

## 2 · Fast loader (with Parquet cache)

First read of a 274 MB CSV is ~1 s; we cache a Parquet copy so re-runs are ~50 ms.
`amplitude` fits in `int32` (values up to ±3e5), `time` is regenerated from the sample
index to avoid float drift.

In [ ]:
def load_signal(row, cache: bool = True, max_seconds: float | None = MAX_SECONDS):
    # Return (t, a) float arrays for one inventory row. a = raw counts (float32).
    src  = Path(row.file)
    pq   = CACHE_DIR / (src.stem + ".parquet")
    if cache and pq.exists():
        a = pd.read_parquet(pq)["a"].to_numpy()
    else:
        a = pd.read_csv(src, header=None, usecols=[1], names=["a"],
                        dtype={"a": "int32"}, engine="c")["a"].to_numpy()
        if cache:
            pd.DataFrame({"a": a}).to_parquet(pq, compression="zstd")
    a = a.astype(np.float32)
    if max_seconds is not None:
        a = a[: int(max_seconds * FS)]
    t = np.arange(a.size, dtype=np.float64) * DT
    return t, a


def decimate_plot(a, max_points: int = 6000):
    # Min/max envelope decimation — keeps spikes visible when downsampling for a plot.
    n = a.size
    if n <= max_points * 2:
        return np.arange(n) * DT, a
    step = int(np.ceil(n / max_points))
    m    = (n // step) * step
    blk  = a[:m].reshape(-1, step)
    lo, hi = blk.min(1), blk.max(1)
    y = np.empty(lo.size * 2, np.float32); y[0::2], y[1::2] = lo, hi
    x = np.repeat(np.arange(lo.size) * step, 2) * DT
    return x, y

In [ ]:
# verify sample rate / timestamp regularity on one raw file
_raw = pd.read_csv(inv.file.iloc[0], header=None, names=["t", "a"], nrows=500_000)
_dt  = np.diff(_raw.t.to_numpy())
print(f"{inv.name.iloc[0]!r}")
print(f"  Δt  mean={_dt.mean():.6e}s  std={_dt.std():.2e}s  min={_dt.min():.3e}  max={_dt.max():.3e}")
print(f"  fs  = {1/_dt.mean():,.1f} Hz   (assumed {FS:,})")
print(f"  monotonic increasing timestamps: {bool(np.all(_dt > 0))}")
del _raw, _dt

## 3 · Load everything once → per-file raw stats + decimated copy for plots

Files are processed one at a time; only the small decimated trace and scalar stats are
kept in memory.

In [ ]:
INT16_MAX = 32767
traces, raw_stats = {}, []

for row in inv.itertuples():
    t0 = time.time()
    t, a = load_signal(row)
    x, y = decimate_plot(a)
    traces[row.name] = (x, y)

    absmax = np.abs(a).max()
    raw_stats.append(dict(
        name=row.name, cls=row.cls, split=row.split, n=a.size,
        min=a.min(), max=a.max(), mean=a.mean(), std=a.std(),
        median=np.median(a), mad=stats.median_abs_deviation(a),
        rms=np.sqrt(np.mean(a**2)), absmax=absmax,
        kurtosis=stats.kurtosis(a), skew=stats.skew(a),
        n_nan=int(np.isnan(a).sum()),
        pct_at_int16_rail=float(np.mean(np.abs(a) >= INT16_MAX) * 100),
        frac_zero=float(np.mean(a == 0) * 100),
        load_s=round(time.time() - t0, 2),
    ))
    del a

raw = pd.DataFrame(raw_stats).set_index("name")
raw.round(2)

## 4 · Data-quality checks

* **NaNs / dtype** — none expected (integer ADC dump).
* **Timestamp regularity** — checked above, uniform 256 kHz.
* **Saturation / clipping** — fraction of samples pinned at the ADC rail.
* **DC offset** — non-zero mean ⇒ subtract in cleaning.
* **Dead / flat segments** — long runs of a constant value (sensor dropout).
* **Duplicate consecutive samples** — sanity only.

In [ ]:
def flat_run_stats(a, tol=0):
    # Longest run of near-constant samples + fraction of samples inside runs >= 256.
    d = np.abs(np.diff(a)) <= tol
    if not d.any():
        return 0, 0.0
    # run-lengths of True in d
    idx = np.flatnonzero(np.diff(np.r_[0, d.view(np.int8), 0]))
    runs = idx[1::2] - idx[0::2]
    long = runs[runs >= 256].sum()
    return int(runs.max()) + 1, float(long / a.size * 100)

qc = []
for row in inv.itertuples():
    _, a = load_signal(row)
    longest, pct_flat = flat_run_stats(a)
    q99 = np.percentile(np.abs(a), 99.99)
    qc.append(dict(
        name=row.name, cls=row.cls,
        n_nan=int(np.isnan(a).sum()),
        dc_offset=float(a.mean()),
        pct_saturated=float(np.mean(np.abs(a) >= 0.999 * np.abs(a).max()) * 100),
        p99_99_abs=float(q99),
        longest_flat_run=longest,
        pct_in_flat_runs=pct_flat,
        n_dup_consec=int(np.sum(np.diff(a) == 0)),
    ))
    del a

qc = pd.DataFrame(qc).set_index("name")
qc.round(3)

In [ ]:
# visual: boxplot of amplitude spread per file (log scale) to spot outlier files
fig, ax = plt.subplots(figsize=(12, 5))
order = inv.sort_values(["split", "cls"]).name.tolist()
data  = []
for nm in order:
    _, a = load_signal(inv[inv.name == nm].iloc[0])
    data.append(np.abs(a[::37]) + 1e-3)   # subsample for the boxplot; avoid log(0)
    del a
ax.boxplot(data, showfliers=False, tick_labels=order)
ax.set_yscale("log"); ax.set_ylabel("|amplitude| (counts, log)")
ax.set_title("Amplitude spread per file  (whiskers = 1.5·IQR, fliers hidden)")
plt.xticks(rotation=75, ha="right"); plt.tight_layout(); plt.show()

## 5 · Cleaning

Pipeline per signal:
1. **DC removal** — subtract the median (robust to PD spikes).
2. **Slow-drift removal** — subtract a 0.1 s moving-median baseline (high-pass ~10 Hz)
   to suppress mechanical / thermal wander while preserving µs-scale PD pulses.
3. Keep counts (no calibration); also expose a **z-scored** copy for cross-file comparison.

Cleaned traces are cached to `../datasets/_processed/<name>.npz`.

In [ ]:
def moving_median(a, win):
    # Fast approximate moving-median baseline: exact median on coarse centres + interp.
    win = int(win)
    if win < 3 or win >= a.size:
        return np.full_like(a, np.median(a))
    step = max(1, win // 4)
    centers = np.arange(0, a.size, step)
    med = np.empty(centers.size, np.float32)
    half = win // 2
    for i, c in enumerate(centers):
        lo, hi = max(0, c - half), min(a.size, c + half)
        med[i] = np.median(a[lo:hi])
    return np.interp(np.arange(a.size), centers, med).astype(np.float32)


def clean_signal(a, fs=FS, drift_win_s=0.1):
    a = a - np.median(a)                              # 1 · DC
    base = moving_median(a, drift_win_s * fs)         # 2 · slow drift
    a = a - base
    z = a / (stats.median_abs_deviation(a) * 1.4826 + 1e-9)
    return a.astype(np.float32), z.astype(np.float32)


clean_stats = []
for row in inv.itertuples():
    _, a = load_signal(row)
    ac, z = clean_signal(a)
    np.savez_compressed(OUT_DIR / f"{row.name}.npz",
                        clean=ac, z=z, fs=FS, cls=row.cls, split=row.split)
    clean_stats.append(dict(name=row.name, cls=row.cls,
                            rms_clean=float(np.sqrt(np.mean(ac**2))),
                            mad_clean=float(stats.median_abs_deviation(ac)),
                            max_abs_z=float(np.abs(z).max())))
    del a, ac, z

pd.DataFrame(clean_stats).set_index("name").round(2)

## 6 · EDA — full 60 s waveforms (min/max decimated)

In [ ]:
order = inv.sort_values(["split", "cls", "name"]).name.tolist()
fig, axes = plt.subplots(len(order), 1, figsize=(13, 1.7 * len(order)), sharex=True)
for ax, nm in zip(axes, order):
    x, y = traces[nm]
    r = inv[inv.name == nm].iloc[0]
    ax.plot(x, y, lw=0.4)
    ax.set_ylabel(f"{r.cls}\n{r.split}", fontsize=7)
    ax.set_title(nm, fontsize=8, loc="left", pad=1)
axes[-1].set_xlabel("time (s)")
fig.suptitle("Full recordings — amplitude (counts) vs time", y=1.001)
plt.tight_layout(); plt.show()

### 6.1 · Zoom on the most active 50 ms window per file

In [ ]:
WIN_S = 0.05
win_n = int(WIN_S * FS)
fig, axes = plt.subplots(4, 3, figsize=(14, 9))
for ax, nm in zip(axes.ravel(), order):
    r = inv[inv.name == nm].iloc[0]
    _, a = load_signal(r)
    a = a - np.median(a)
    # locate window of maximum energy
    e = np.add.reduceat(a.astype(np.float64) ** 2,
                        np.arange(0, a.size, win_n))
    k = int(np.argmax(e[:-1]))
    seg = a[k * win_n:(k + 1) * win_n]
    ts  = np.arange(seg.size) / FS * 1e3
    ax.plot(ts, seg, lw=0.6)
    ax.set_title(f"{nm}\n@ t≈{k*WIN_S:5.1f}s", fontsize=7)
    ax.set_xlabel("ms", fontsize=7)
    del a
for ax in axes.ravel()[len(order):]:
    ax.axis("off")
fig.suptitle(f"Most energetic {int(WIN_S*1e3)} ms window per file", y=1.01)
plt.tight_layout(); plt.show()

### 6.2 · Amplitude distributions (cleaned, per class)

In [ ]:
classes = sorted(inv.cls.unique())
fig, axes = plt.subplots(1, len(classes), figsize=(3.6 * len(classes), 3.4), sharey=True)
for ax, c in zip(np.atleast_1d(axes), classes):
    for nm in inv[inv.cls == c].name:
        d = np.load(OUT_DIR / f"{nm}.npz")
        a = d["clean"][::11]
        ax.hist(a, bins=200, histtype="step", lw=1,
                label=nm[:22], density=True, log=True)
    ax.set_title(c); ax.set_xlabel("amplitude (counts)")
    ax.legend(fontsize=6)
axes[0].set_ylabel("density (log)")
fig.suptitle("Cleaned amplitude histograms by defect class", y=1.03)
plt.tight_layout(); plt.show()

### 6.3 · Envelope / short-term RMS over the full record

In [ ]:
RMS_WIN_S = 0.02
w = int(RMS_WIN_S * FS)
fig, axes = plt.subplots(len(classes), 1, figsize=(13, 2.1 * len(classes)), sharex=True)
for ax, c in zip(np.atleast_1d(axes), classes):
    for nm in inv[inv.cls == c].name:
        d = np.load(OUT_DIR / f"{nm}.npz")
        a = d["clean"].astype(np.float64)
        m = (a.size // w) * w
        rms = np.sqrt(np.mean(a[:m].reshape(-1, w) ** 2, axis=1))
        tt  = np.arange(rms.size) * RMS_WIN_S
        ax.plot(tt, rms, lw=0.7, label=nm[:26])
        del a
    ax.set_ylabel(f"{c}\nRMS"); ax.legend(fontsize=6, ncol=2)
axes[-1].set_xlabel("time (s)")
fig.suptitle(f"Short-term RMS envelope ({int(RMS_WIN_S*1e3)} ms windows)", y=1.005)
plt.tight_layout(); plt.show()

### 6.4 · Power spectral density (Welch) — class comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
cmap = plt.cm.tab10
for i, c in enumerate(classes):
    for j, nm in enumerate(inv[inv.cls == c].name):
        d = np.load(OUT_DIR / f"{nm}.npz")
        f, pxx = signal.welch(d["clean"], fs=FS, nperseg=8192)
        ax.semilogy(f / 1e3, pxx, color=cmap(i), lw=0.9,
                    alpha=0.9, label=c if j == 0 else None)
ax.set_xlabel("frequency (kHz)"); ax.set_ylabel("PSD (counts²/Hz)")
ax.set_title("Welch PSD — colour = defect class"); ax.legend()
plt.tight_layout(); plt.show()

### 6.5 · Spectrograms (STFT) per file

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(15, 10), sharex=True, sharey=True)
for ax, nm in zip(axes.ravel(), order):
    d = np.load(OUT_DIR / f"{nm}.npz")
    f, tt, Sxx = signal.spectrogram(d["clean"], fs=FS, nperseg=2048, noverlap=1024)
    ax.pcolormesh(tt, f / 1e3, 10 * np.log10(Sxx + 1e-6),
                  shading="auto", cmap="magma")
    ax.set_title(nm[:30], fontsize=7)
for ax in axes[-1]:
    ax.set_xlabel("time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel("kHz")
for ax in axes.ravel()[len(order):]:
    ax.axis("off")
fig.suptitle("Spectrograms  (10·log10 power, dB)", y=1.01)
plt.tight_layout(); plt.show()

## 7 · PD pulse detection

Threshold on the robust z-score (`|z| > K`), merge samples within a refractory window
into single events, then characterise event **rate**, **amplitude** and
**inter-event interval**.

In [ ]:
K_THRESH   = 6.0        # robust-z detection threshold
REFRACT_S  = 5e-4       # merge detections closer than this into one pulse

def detect_pulses(z, fs=FS, k=K_THRESH, refract_s=REFRACT_S):
    over = np.flatnonzero(np.abs(z) > k)
    if over.size == 0:
        return np.empty(0, int), np.empty(0)
    gap = int(refract_s * fs)
    grp = np.split(over, np.flatnonzero(np.diff(over) > gap) + 1)
    idx = np.array([g[np.argmax(np.abs(z[g]))] for g in grp])
    amp = z[idx]
    return idx, amp

pulse_rows = []
for row in inv.itertuples():
    d = np.load(OUT_DIR / f"{row.name}.npz")
    z = d["z"]
    idx, amp = detect_pulses(z)
    dur = z.size / FS
    ipi = np.diff(idx) / FS if idx.size > 1 else np.array([np.nan])
    pulse_rows.append(dict(
        name=row.name, cls=row.cls, split=row.split,
        n_pulses=idx.size, rate_hz=idx.size / dur,
        amp_med=float(np.median(np.abs(amp))) if amp.size else np.nan,
        amp_p95=float(np.percentile(np.abs(amp), 95)) if amp.size else np.nan,
        ipi_med_ms=float(np.median(ipi) * 1e3),
        ipi_cv=float(np.std(ipi) / np.mean(ipi)) if idx.size > 2 else np.nan,
    ))
    del z

pulses = pd.DataFrame(pulse_rows).set_index("name")
pulses.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
colors = {c: plt.cm.tab10(i) for i, c in enumerate(classes)}
p = pulses.sort_values(["split", "cls"])
ax[0].bar(range(len(p)), p.rate_hz, color=[colors[c] for c in p.cls])
ax[0].set_xticks(range(len(p))); ax[0].set_xticklabels(p.index, rotation=75, ha="right")
ax[0].set_ylabel("pulses / s"); ax[0].set_title(f"PD pulse rate  (|z| > {K_THRESH})")

for c in classes:
    sub = pulses[pulses.cls == c]
    ax[1].scatter(sub.rate_hz, sub.amp_p95, s=60, color=colors[c], label=c)
ax[1].set_xlabel("pulse rate (Hz)"); ax[1].set_ylabel("pulse amplitude p95 (robust-z)")
ax[1].set_xscale("log"); ax[1].set_title("rate vs amplitude"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8 · Per-file feature table

In [ ]:
BANDS = [(0, 20e3), (20e3, 50e3), (50e3, 80e3), (80e3, 128e3)]

def features(a, fs=FS):
    a = a.astype(np.float64)
    rms = np.sqrt(np.mean(a**2))
    peak = np.abs(a).max()
    f, pxx = signal.welch(a, fs=fs, nperseg=8192)
    psd_n = pxx / pxx.sum()
    cent  = float(np.sum(f * psd_n))
    bw    = float(np.sqrt(np.sum((f - cent) ** 2 * psd_n)))
    band_e = {f"band_{int(lo/1e3)}_{int(hi/1e3)}k":
              float(pxx[(f >= lo) & (f < hi)].sum() / pxx.sum()) for lo, hi in BANDS}
    zcr = float(np.mean(np.abs(np.diff(np.sign(a))) > 0))
    return dict(
        rms=rms, peak=peak, crest=peak / (rms + 1e-9),
        kurtosis=float(stats.kurtosis(a)), skew=float(stats.skew(a)),
        spec_centroid_hz=cent, spec_bandwidth_hz=bw,
        dom_freq_hz=float(f[np.argmax(pxx)]), zcr=zcr, **band_e,
    )

feat_rows = []
for row in inv.itertuples():
    d = np.load(OUT_DIR / f"{row.name}.npz")
    fr = features(d["clean"])
    fr.update(name=row.name, cls=row.cls, split=row.split)
    feat_rows.append(fr)

feat = pd.DataFrame(feat_rows).set_index("name")
feat = feat.join(pulses[["n_pulses", "rate_hz", "amp_p95", "ipi_med_ms", "ipi_cv"]])
feat.to_csv(OUT_DIR / "features.csv")
print("saved", OUT_DIR / "features.csv")
feat.round(3)

In [ ]:
# standardised feature heatmap (train vs test grouped)
from matplotlib.colors import TwoSlopeNorm
num = feat.select_dtypes("number")
zz  = (num - num.mean()) / num.std()
rows = feat.sort_values(["cls", "split"]).index
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(zz.loc[rows].T, aspect="auto", cmap="RdBu_r",
               norm=TwoSlopeNorm(0, -2.5, 2.5))
ax.set_xticks(range(len(rows))); ax.set_xticklabels(rows, rotation=75, ha="right")
ax.set_yticks(range(zz.shape[1])); ax.set_yticklabels(zz.columns, fontsize=7)
fig.colorbar(im, label="z-score"); ax.set_title("Standardised features per file")
plt.tight_layout(); plt.show()

In [ ]:
# 2-D PCA of the standardised features, coloured by class
from sklearn.decomposition import PCA
X = zz.fillna(0).to_numpy()
pc = PCA(n_components=2).fit(X)
Y  = pc.transform(X)
fig, ax = plt.subplots(figsize=(7, 6))
for c in classes:
    mask = (feat["cls"] == c).to_numpy()
    ax.scatter(Y[mask, 0], Y[mask, 1], s=90, label=c)
for (x, y), nm, sp in zip(Y, feat.index, feat["split"]):
    ax.annotate(f"{nm[:14]} ({sp})", (x, y), fontsize=6,
                xytext=(4, 3), textcoords="offset points")
ax.set_xlabel(f"PC1 ({pc.explained_variance_ratio_[0]:.0%})")
ax.set_ylabel(f"PC2 ({pc.explained_variance_ratio_[1]:.0%})")
ax.set_title("Feature-space PCA"); ax.legend()
plt.tight_layout(); plt.show()

## 9 · Windowed export for modelling

Slice every cleaned recording into non-overlapping `WINDOW_S` windows and stack them
into `X` (`n_windows × window_len`) with aligned `y` (class) and `groups` (source file),
saved to `../datasets/_processed/windows.npz`. Ready for a classifier or for
`foreblocks` sequence models.

In [ ]:
WINDOW_S   = 1.0
win_len    = int(WINDOW_S * FS)

X, y, groups, split_arr = [], [], [], []
for row in inv.itertuples():
    d = np.load(OUT_DIR / f"{row.name}.npz")
    a = d["z"]                      # robust-z scaled -> comparable across files
    m = (a.size // win_len) * win_len
    W = a[:m].reshape(-1, win_len).astype(np.float32)
    X.append(W)
    y       += [row.cls] * W.shape[0]
    groups  += [row.name] * W.shape[0]
    split_arr += [row.split] * W.shape[0]
    del a

X = np.concatenate(X, 0)
y = np.array(y); groups = np.array(groups); split_arr = np.array(split_arr)
np.savez_compressed(OUT_DIR / "windows.npz",
                    X=X, y=y, groups=groups, split=split_arr,
                    fs=FS, window_s=WINDOW_S)
print("windows.npz  X:", X.shape, " classes:", dict(zip(*np.unique(y, return_counts=True))))
print("train windows:", int((split_arr == 'train').sum()),
      " test windows:", int((split_arr == 'test').sum()))

## 10 · Summary / next steps

* **Data is clean** — no NaNs, uniform 256 kHz sampling, integer ADC counts. Main
  preprocessing needs: DC-offset removal + slow-drift high-pass (done in §5). Check the
  QC table for any file flagged with saturation or long flat runs.
* **Class signatures** — compare the Welch PSD (§6.4), spectrograms (§6.5) and pulse-rate
  chart (§7): `dba` shows the highest energy / broadband content, `corona` the lowest,
  `triplen_point` / `surface_discharge` sit between with distinct pulse timing.
* **Artifacts written** to `../datasets/_processed/`:
  `*.npz` (cleaned + z-scored traces), `features.csv`, `windows.npz` (ML-ready).
* **Next** — phase-resolved PD (PRPD) patterns need a synchronous HV reference channel
  (not in this dataset); with only the acoustic channel, use the windowed features /
  raw windows for a supervised classifier, keeping `split` = train/test as provided.